In [1]:
!pip install minsearch


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
import json
from openai import OpenAI
from minsearch import AppendableIndex

# 1. Descargamos e indexamos los documentos del curso (Usaremos AppendableIndex para poder agregarle cosas luego)
print("Descargando e indexando documentos...")
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []
for course in documents_raw:
    course_name = course['course']
    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(documents)

# 2. Función de búsqueda clásica
def search(query):
    boost = {'question': 3.0, 'section': 0.5}
    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )
    return results

# 3. Cliente de OpenAI y función base
client = OpenAI()

def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 4. ¡EL NUEVO PROMPT AGÉNTICO!
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}
""".strip()

print("¡Sistema listo para hacer de Agente!")

Descargando e indexando documentos...
¡Sistema listo para hacer de Agente!


In [3]:
# Hacemos la pregunta de prueba
question = 'Can I still join the course?'
context = 'EMPTY'

# Armamos el prompt y le preguntamos al LLM
prompt = prompt_template.format(question=question, context=context)
answer_json = llm(prompt)

# Parseamos la respuesta JSON del LLM
respuesta_diccionario = json.loads(answer_json)

print("--- DECISIÓN DEL AGENTE ---")
print(json.dumps(respuesta_diccionario, indent=2))
print(f"\nAcción elegida por el LLM: {respuesta_diccionario['action']}")

--- DECISIÓN DEL AGENTE ---
{
  "action": "SEARCH",
  "reasoning": "The question is about joining the course, and without any context provided, I need to refer to the FAQ database for information regarding enrollment opportunities."
}

Acción elegida por el LLM: SEARCH


In [4]:
# 1. Función auxiliar para armar el texto del contexto
def build_context(search_results):
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    return context.strip()

# 2. Ejecutamos la búsqueda que el LLM solicitó
resultados_busqueda = search(question)
contexto_lleno = build_context(resultados_busqueda)

# 3. Volvemos a llamar al LLM, pasándole el contexto lleno
prompt_con_datos = prompt_template.format(question=question, context=contexto_lleno)
respuesta_final_json = llm(prompt_con_datos)

print("--- RESPUESTA FINAL DEL AGENTE ---")
print(json.dumps(json.loads(respuesta_final_json), indent=2))

--- RESPUESTA FINAL DEL AGENTE ---
{
  "action": "ANSWER",
  "answer": "Yes, you can still join the course even after the start date. Although it's recommended to register, you are still eligible to submit homework and engage with the course materials. Keep in mind that there will be deadlines for the final projects, so it's best not to procrastinate.",
  "source": "CONTEXT"
}


In [5]:
# 1. Función para eliminar documentos duplicados de múltiples búsquedas
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

# 2. El "Súper Prompt" Investigador
prompt_agente_investigador = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2"]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

# 3. El Bucle (While Loop) de Investigación
question = "what do I need to do to be successful at module 1?"

search_queries = []
search_results = []
previous_actions = []
iteration = 0
max_iterations = 3

print(f"Investigando la pregunta: '{question}'...\n")

while True:
    print(f"--- ITERACIÓN #{iteration} ---")
    
    # Preparamos los datos actuales
    context = build_context(search_results)
    prompt = prompt_agente_investigador.format(
        question=question,
        context=context,
        search_queries="\n".join(search_queries),
        previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
        max_iterations=max_iterations,
        iteration_number=iteration
    )
    
    # Consultamos al LLM
    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    
    print(f"Acción decidida por el bot: {answer['action']}")
    
    # Guardamos la acción en la memoria
    previous_actions.append(answer)
    
    # Si la acción NO es buscar, significa que ya tiene la respuesta. Rompemos el ciclo.
    if answer['action'] != 'SEARCH':
        print("\n✅ ¡RESPUESTA ENCONTRADA!")
        print(answer['answer'])
        break
        
    # Si la acción ES buscar, extraemos las palabras clave y buscamos
    keywords = answer['keywords']
    print(f"Palabras clave a buscar: {keywords}\n")
    search_queries = list(set(search_queries) | set(keywords))
    
    for k in keywords:
        res = search(k)
        search_results.extend(res)
        
    # Limpiamos duplicados
    search_results = dedup(search_results)
    
    iteration += 1
    if iteration >= max_iterations:
        print("\nLímite máximo de iteraciones alcanzado. Deteniendo.")
        break

Investigando la pregunta: 'what do I need to do to be successful at module 1?'...

--- ITERACIÓN #0 ---
Acción decidida por el bot: SEARCH
Palabras clave a buscar: ['successful module 1', 'tips for module 1', 'module 1 guidelines']

--- ITERACIÓN #1 ---
Acción decidida por el bot: SEARCH
Palabras clave a buscar: ['how to succeed in Module 1', 'Module 1 study tips', 'Module 1 success strategies']

--- ITERACIÓN #2 ---
Acción decidida por el bot: ANSWER

✅ ¡RESPUESTA ENCONTRADA!
To be successful in Module 1, focus on understanding the essential concepts of Docker and Terraform, as they form the backbone of this module. Here are some tips: 
1. **Follow the course materials closely**: Pay attention to lecture videos, documentation, and any additional resources provided by the instructor. 
2. **Practice hands-on**: Engage with practical exercises and use Docker and Terraform to set up your environments. Hands-on experience is crucial for grasping the concepts. 
3. **Seek help when needed**:

In [6]:
# 1. Definimos la estructura de nuestra Herramienta (Tool) para OpenAI
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# 2. Función auxiliar que ejecuta la herramienta solicitada por el LLM
def do_call(tool_call_response):
    # Extraemos el nombre de la función y los argumentos que el LLM nos pide usar
    function_name = tool_call_response.name
    arguments = json.loads(tool_call_response.arguments)

    # globals()[function_name] busca en la memoria de Python una función que se llame así
    f = globals()[function_name]
    
    # Ejecutamos la función (en este caso, 'search(query=...)')
    result = f(**arguments)

    # Devolvemos el resultado empaquetado para que el LLM lo entienda
    return {
        "type": "function_call_output",
        "call_id": tool_call_response.call_id,
        "output": json.dumps(result, indent=2),
    }

print("¡Herramienta registrada exitosamente!")

¡Herramienta registrada exitosamente!


In [7]:
question = "How do I do well in module 1?"

# Fíjate lo corto y simple que es el prompt ahora
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
If you look up something in FAQ, convert the student question into multiple queries.
""".strip()

# Le pasamos la lista de herramientas disponibles
tools = [search_tool]

# Historial de mensajes
chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

print("Consultando al LLM...\n")

# Hacemos la llamada a OpenAI usando client.responses.create
response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)

# Imprimimos lo que nos devolvió el modelo
calls = response.output
print("Lo que respondió el LLM:\n")
for call in calls:
    print(call)

Consultando al LLM...

Lo que respondió el LLM:

ResponseFunctionToolCall(arguments='{"query":"how to do well in module 1"}', call_id='call_pCZMeZlhTVtSH6qVmvZ5W20c', name='search', type='function_call', id='fc_060418678d989c55006a259b8cbf9c819f9e73126cf7b34586', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"study tips for module 1"}', call_id='call_TjkTEIwlNPRg4ght1GbtGmlW', name='search', type='function_call', id='fc_060418678d989c55006a259b8cbfb0819fa8c9661d06fb5178', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"module 1 resources"}', call_id='call_PUG2jIKqSO0EdMSFD2FGwV2O', name='search', type='function_call', id='fc_060418678d989c55006a259b8cbfbc819fa5e736271aebd8f9', namespace=None, status='completed')


In [8]:
# 1. Procesamos cada llamada a la herramienta que nos pidió el LLM
for call in calls:
    # Ejecutamos la búsqueda internamente usando nuestra función auxiliar
    result = do_call(call)
    
    # Añadimos la petición original del LLM al historial de mensajes
    chat_messages.append(call)
    # Añadimos los resultados de nuestra búsqueda al historial
    chat_messages.append(result)

print("Resultados de búsqueda agregados al historial. Volviendo a consultar al LLM...\n")

# 2. Volvemos a llamar a OpenAI, pero ahora el historial incluye los documentos encontrados
response_final = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)

# 3. Imprimimos la respuesta final
for mensaje in response_final.output:
    if mensaje.type == 'message':
        print("--- RESPUESTA FINAL DEL AGENTE ---")
        print(mensaje.content[0].text)

Resultados de búsqueda agregados al historial. Volviendo a consultar al LLM...

--- RESPUESTA FINAL DEL AGENTE ---
To do well in Module 1, here are some strategies and resources:

### Study Tips:
1. **Understand Key Concepts**: Make sure you grasp the fundamental concepts covered in the module. This includes understanding Docker and Terraform basics.
2. **Practice Regularly**: Apply what you learn through hands-on exercises. Set up Docker containers and work with Terraform to solidify your knowledge.
3. **Engage with Course Resources**: Utilize any provided resources, such as video lectures, readings, and supplementary materials.
4. **Join Study Groups**: Collaborate with classmates to discuss topics and tackle challenging concepts together.
5. **Reach Out for Help**: Don’t hesitate to ask questions in forums or reach out to instructors/teaching assistants if you’re struggling with specific topics.

### Recommended Resources:
- **Error Handling**: Be familiar with common errors and how

In [11]:
!pip install markdown


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [12]:
import urllib.request
import json

# 1. PRIMERO descargamos el script refactorizado del profesor
print("Descargando chat_assistant.py...")
url_script = 'https://raw.githubusercontent.com/alexeygrigorev/rag-agents-workshop/refs/heads/main/chat_assistant.py'
urllib.request.urlretrieve(url_script, 'chat_assistant.py')

# 2. AHORA SÍ lo importamos (porque el archivo ya existe en tu entorno)
import chat_assistant

# 3. Definimos la nueva función de Python para agregar datos
def add_entry(question, answer):
    doc = {
        'question': question,
        'text': answer,
        'section': 'user added',
        'course': 'data-engineering-zoomcamp'
    }
    index.append(doc)

# 4. Definimos la descripción (El "Menú") de la nueva herramienta
add_entry_description = {
    "type": "function",
    "name": "add_entry",
    "description": "Add an entry to the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question to be added to the FAQ database",
            },
            "answer": {
                "type": "string",
                "description": "The answer to the question",
            }
        },
        "required": ["question", "answer"],
        "additionalProperties": False
    }
}

# 5. Registramos AMBAS herramientas usando la clase Tools del profesor
tools = chat_assistant.Tools()
tools.add_tool(search, search_tool)
tools.add_tool(add_entry, add_entry_description)

print("Herramientas registradas:", [t['name'] for t in tools.get_tools()])

# 6. Preparamos el Prompt y la Interfaz
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

chat_interface = chat_assistant.ChatInterface()

# 7. Inicializamos el Asistente
chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client
)

print("\n¡Bot iniciado! Escribe tu pregunta abajo. Escribe 'stop' para salir.")

# 8. Ejecutamos el bucle principal
chat.run()

Descargando chat_assistant.py...
Herramientas registradas: ['search', 'add_entry']

¡Bot iniciado! Escribe tu pregunta abajo. Escribe 'stop' para salir.


Chat ended.


In [13]:
index.docs[-1]

{'question': 'How do I do well in module 1?',
 'text': "To do well in Module 1, consider the following strategies:\n\n1. **Understand the Concepts**: Make sure you have a solid grasp of the foundational concepts taught in this module. Review your lecture notes and any provided resources.\n\n2. **Practice Hands-On**: Engage in practical exercises related to the material. This could include coding tasks or hands-on projects.\n\n3. **Utilize Resources**: Don’t hesitate to consult supplementary resources like textbooks, online tutorials, or recorded lectures to reinforce your understanding.\n\n4. **Collaborate with Peers**: Joining study groups or discussion forums can provide new insights and support.\n\n5. **Ask Questions**: If you're stuck or confused about a topic, reach out to your instructors or fellow students for clarification.\n\n6. **Review Feedback**: Pay attention to any feedback you receive on assignments or quizzes to understand where you can improve.",
 'section': 'user adde